In [1]:
import sqlite3
import re
import pandas as pd

In [2]:
!pip install pyodbc




Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pyodbc
print(pyodbc.drivers())



['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Oracle in OraDB21Home1', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [4]:
import pyodbc

# Paramètres de connexion
server = 'SARAH_BHSS\\SQLEXPRESS'  # Nom du serveur SQL avec instance
database = 'SystemeSuiviProduction'  # Nom de la base de données
username = 'sa'  # Nom d'utilisateur SQL
password = 'admin'  # Mot de passe de l'utilisateur

# Connexion à SQL Server
try:
    conn = pyodbc.connect(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={username};"
        f"PWD={password};"
    )
    print("Connexion réussie !")
except Exception as e:
    print("Erreur lors de la connexion :", e)


Connexion réussie !


# Connexion avec la base Oracle  => Importation Oracle vers CSV

In [5]:
pip install cx_Oracle


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import cx_Oracle
import pandas as pd

# Définir les informations de connexion
dsn = cx_Oracle.makedsn("localhost", 1521, service_name="xe")

try:
    # Connexion à la base de données
    connection = cx_Oracle.connect(user="system", password="admin", dsn=dsn)
    print("Connexion réussie à la base de données.")

    # Exécuter une requête SQL
    query = "SELECT * FROM BESOIN_OF_GROF_H"
    data = pd.read_sql(query, con=connection)

    # Exporter les données dans un fichier CSV
    output_path = "C:/dmp_files/BESOIN_OF_GROF_H.csv"
    data.to_csv(output_path, index=False)
    print(f"Données exportées avec succès dans {output_path}")

except cx_Oracle.DatabaseError as e:
    print(f"Erreur lors de la connexion ou de l'exécution : {e}")

finally:
    # Fermer la connexion
    if 'connection' in locals() and connection:
        connection.close()
        print("Connexion fermée.")


Connexion réussie à la base de données.


C:\Users\sarah\AppData\Local\Temp\ipykernel_35404\2701388935.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, con=connection)


Données exportées avec succès dans C:/dmp_files/BESOIN_OF_GROF_H.csv
Connexion fermée.


In [7]:
import cx_Oracle
import pandas as pd

# Définir les informations de connexion
dsn = cx_Oracle.makedsn("localhost", 1521, service_name="xe")

try:
    # Connexion à la base de données
    connection = cx_Oracle.connect(user="system", password="admin", dsn=dsn)
    print("Connexion réussie à la base de données.")

    # Exécuter une requête SQL
    query = "SELECT * FROM BESOIN_OF_H"
    data = pd.read_sql(query, con=connection)

    # Exporter les données dans un fichier CSV
    output_path = "C:/dmp_files/BESOIN_OF_H.csv"
    data.to_csv(output_path, index=False)
    print(f"Données exportées avec succès dans {output_path}")

except cx_Oracle.DatabaseError as e:
    print(f"Erreur lors de la connexion ou de l'exécution : {e}")

finally:
    # Fermer la connexion
    if 'connection' in locals() and connection:
        connection.close()
        print("Connexion fermée.")


Connexion réussie à la base de données.


C:\Users\sarah\AppData\Local\Temp\ipykernel_35404\163135212.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, con=connection)


Données exportées avec succès dans C:/dmp_files/BESOIN_OF_H.csv
Connexion fermée.


# Ajout "NumMachine"

In [8]:
import os

def parse_and_combine_records(file_path):
    """
    Parse the SDC file and combine sections like 'LearnStarted' with the following 'Counter' section.
    """
    excluded_blocks = ["MaterialChangeDetection", "MeasurementsLeadSet1", "QualityParameters", "MeasurementData"]

    # Extraire le numéro de la machine à partir du chemin
    # On remonte trois niveaux dans le chemin pour récupérer "433"
    num_machine = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(file_path))))

    with open(file_path, 'r') as f:
        transactions = []
        current_record = {}
        current_section = None

        for line in f:
            line = line.strip()

            if line.startswith("[") and line.endswith("]"):
                # Ajouter l'enregistrement précédent avant de passer à un nouveau bloc
                if current_record:
                    transactions.append(current_record)

                # Initialiser une nouvelle section
                current_section = line[1:-1]
                if current_section not in excluded_blocks:
                    current_record = {"Section": current_section}
                else:
                    current_record = {}
            elif "=" in line and current_section not in excluded_blocks:
                key, value = line.split("=", 1)
                value = value.strip()

                # Parsing des champs spécifiques
                if key == "DateTimeStamp":
                    date, time = value.split(",")
                    current_record["Date"] = date.strip()
                    current_record["Time"] = time.strip()
                elif "," in value:
                    parts = value.split(",")
                    if key == "Wire":
                        current_record["Wire Code"] = parts[0].strip()
                        current_record["Wire Quantity Global"] = parts[1].strip()
                        current_record["Wire Good Quantity"] = parts[2].strip()
                    elif key == "Terminal":
                        current_record["Terminal Code"] = parts[0].strip()
                        current_record["Terminal Quantity Global"] = parts[1].strip()
                        current_record["Terminal Good Quantity"] = parts[2].strip()
                    elif key == "Seal":
                        current_record["Seal Code"] = parts[0].strip()
                        current_record["Seal Quantity Global"] = parts[1].strip()
                        current_record["Seal Good Quantity"] = parts[2].strip()
                    elif key == "Job":
                        current_record["Job"] = f"{parts[0].strip()},{parts[1].strip()}"
                    elif key == "ProductionPieces":
                        current_record["ProductionGoodPieces"] = parts[0].strip()
                        current_record["ProductionPieces (Demandé)"] = parts[1].strip()
                else:
                    current_record[key.strip()] = value

        # Ajouter le dernier enregistrement
        if current_record:
            transactions.append(current_record)

    # Fusionner LearnStarted avec le Counter qui suit immédiatement
    merged_transactions = []
    temp_record = {}

    for record in transactions:
        if record["Section"] != "Counter":
            if temp_record:
                merged_transactions.append(temp_record)
            temp_record = record.copy()
        else:
            if temp_record:
                for key, value in record.items():
                    if key not in temp_record or not temp_record[key]:
                        temp_record[key] = value
                merged_transactions.append(temp_record)
                temp_record = {}

    if temp_record:
        merged_transactions.append(temp_record)

    # Ajouter NumMachine à chaque enregistrement
    for record in merged_transactions:
        record["NumMachine"] = num_machine

    # Convertir en DataFrame
    df = pd.DataFrame(merged_transactions)

    # Sauvegarder en CSV
    output_file = os.path.join(os.path.dirname(file_path), 'combined_Producti_Machine.csv')
    df.to_csv(output_file, index=False, encoding='utf-8')

    print(f"Le fichier combiné a été sauvegardé avec succès à l'emplacement : {output_file}")
    return df


# Exemple d'appel de la fonction parse_sdc_file avec un fichier SDC spécifique
file_path = r"C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\12\ProductiTXT.TXT"
# Appeler la fonction de parsing
df = parse_and_combine_records(file_path)


Le fichier combiné a été sauvegardé avec succès à l'emplacement : C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\12\combined_Producti_Machine.csv


In [9]:
df

,Section,Date,Time,ArticleKey,Job,UserName,SampleRequestedPieces,Wire Code,Wire Quantity Global,Wire Good Quantity,Terminal Code,Terminal Quantity Global,Terminal Good Quantity,NumMachine,ProductionRequestedPieces,TotalGoodPieces,UserRequestedPieces
0,SampleStarted,12/11/2024,06:34:14,58002_-39,"58002_-39,1",MED AMIN,4,CU04CR041,2206627,2106175,T927771-3,3086,2568,433,NaN,NaN,NaN
1,SampleAborted,12/11/2024,06:34:17,58002_-39,"58002_-39,1",MED AMIN,4,CU04CR041,2206627,2106175,T927771-3,3086,2568,433,NaN,NaN,NaN
2,SampleStarted,12/11/2024,06:35:57,58002_-39,"58002_-39,1",MED AMIN,1,CU04CR041,2206627,2106175,T927771-3,3086,2568,433,NaN,NaN,NaN
3,SampleTerminated,12/11/2024,06:36:09,58002_-39,"58002_-39,1",MED AMIN,1,CU04CR041,2206945,2106175,T927771-3,3087,2568,433,NaN,NaN,NaN
4,SampleStarted,12/11/2024,06:38:37,58002_-39,"58002_-39,1",MED AMIN,1,CU04CR041,2206945,2106175,T927771-3,3087,2568,433,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,SampleTerminated,12/11/2024,14:13:23,57260_-40,"57260_-40,1",MED AMIN,1,CU08CR003,6328701,6047803,T154717-3,5325,4657,433,NaN,NaN,NaN
251,LearnStarted,12/11/2024,14:14:11,57260_-40,"57260_-40,1",MED AMIN,NaN,CU08CR003,6328701,6047803,T154717-3,5325,4657,433,NaN,NaN,NaN
252,LearnTerminated,12/11/2024,14:14:35,57260_-40,"57260_-40,1",MED AMIN,NaN,CU08CR003,6329321,6047803,T154717-3,5328,4657,433,NaN,NaN,NaN
253,SampleStarted,12/11/2024,14:16:59,57260_-40,"57260_-40,1",MED AMIN,1,CU08CR003,6329321,6047803,T154717-3,5328,4657,433,NaN,NaN,NaN


# Chargement "Job"

In [10]:
'''import os
import pandas as pd
import pyodbc

# Définir les répertoires source et de sortie
source_folder =  r"C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax"

# Paramètres de connexion
server = 'SARAH_BHSS\\SQLEXPRESS'
database = 'SystemeSuiviProduction'
username = 'sa'
password = 'admin'

# Fonction pour établir la connexion à SQL Server
def connect_to_db():
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password};"
        )
        print("Connexion réussie à la base de données.")
        return conn
    except Exception as e:
        print(f"Erreur lors de la connexion : {e}")
        return None

# Fonction pour insérer un DataFrame dans SQL Server
def insert_data_to_sql(df, table_name, conn):
    try:
        cursor = conn.cursor()
        # Vérifier si le DataFrame est vide
        if df.empty:
            print(f"Le DataFrame est vide. Aucune donnée à insérer pour la table '{table_name}'.")
            return
        # Générer la requête d'insertion SQL
        columns = ', '.join(df.columns)
        placeholders = ', '.join(['?'] * len(df.columns))
        insert_query = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
        # Insérer chaque ligne du DataFrame dans la table
        for _, row in df.iterrows():
            cursor.execute(insert_query, tuple(row))
        conn.commit()
        print(f"Données insérées avec succès dans la table '{table_name}'.")
    except Exception as e:
        print(f"Erreur lors de l'insertion des données : {e}")

# Fonction pour parser les fichiers IDC et LDC
def parse_data_file(file_path):
    try:
        # Lire le fichier avec des tabulations comme séparateurs, en remplaçant les 'NULL' par des chaînes vides
        df = pd.read_csv(file_path, sep='\t', header=None, engine='python', na_values=["NULL"])
        # Vérifier le nombre de colonnes
        print(f"Nombre de colonnes dans le fichier : {df.shape[1]}")
        # Définir les noms des colonnes en fonction du nombre de colonnes
        if df.shape[1] == 7:
            df.columns = ['Date', 'Time', 'Source', 'Action', 'Details', 'Status', 'Type']
        elif df.shape[1] == 6:
            df.columns = ['Date', 'Time', 'Source', 'Action', 'Details', 'Status']
        else:
            print(f"Le fichier '{file_path}' a un nombre de colonnes inattendu.")
            return pd.DataFrame()
        print(f"Fichier '{file_path}' chargé avec succès.")

        # Convertir les colonnes Date et Time en formats appropriés
        df['DateJob'] = pd.to_datetime(df['Date'], format='%d/%m/%Y').dt.date
        df['TimeJob'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time

        # Récupérer d'autres colonnes si nécessaire
        df['MainWPCSS'] = df['Source']
        df['ModeJob'] = df['Action']
        df['IdJob'] = df['Details']
        df['NumJob'] = df['Status']
        df['NumJob'] = '1'
        
        # Rendre 'TypeJob' constant, 'A' si 'ModeJob' est 'A', sinon '0'
        df['TypeJob'] = '0'
        
        # Sélectionner les colonnes correspondantes à la table JOB433
        df = df[['DateJob', 'TimeJob', 'MainWPCSS', 'ModeJob', 'IdJob', 'NumJob', 'TypeJob']]
        
        # Supprimer les lignes contenant des valeurs manquantes
        df = df.dropna()
        print(f"Fichier '{file_path}' parsé avec succès.")
        return df
    except Exception as e:
        print(f"Erreur lors du parsing du fichier '{file_path}' : {e}")
        return pd.DataFrame()

# Fonction principale pour traiter les fichiers
def process_files_and_insert():
    # Connexion à la base de données
    conn = connect_to_db()
    if not conn:
        return
    # Parcourir les fichiers dans le répertoire source
    for root, dirs, files in os.walk(source_folder):
        for file in files:
            file_path = os.path.join(root, file)
            if file.endswith('.ldc'):  # Vérification insensible à la casse
                print(f"Fichier trouvé : {file_path}")
                try:
                    # Lire le fichier en DataFrame
                    df = parse_data_file(file_path)
                    # Vérifier si le DataFrame est vide
                    if df.empty:
                        print(f"Le fichier '{file_path}' ne contient aucune donnée valide.")
                        continue  # Passer au fichier suivant
                    # Extraire la valeur avant la virgule dans 'IdJob'
                    df['IdJob'] = df['IdJob'].str.split(',').str[0]  # Garde la partie avant la virgule
                    # Ajouter une colonne pour tracer la source des données
                    df['FilePath'] = file_path
                    # Supprimer la colonne 'FilePath' avant l'insertion dans SQL
                    df = df.drop(columns=['FilePath'], errors='ignore')
                    # Insérer les données dans la table SQL
                    table_name = 'JOB'  # Nom de la table cible
                    insert_data_to_sql(df, table_name, conn)
                except Exception as e:
                    print(f"Erreur lors du traitement du fichier '{file_path}' : {e}")
    # Fermer la connexion à la base de données
    conn.close()

# Exécuter le script principal
if __name__ == "__main__":
    process_files_and_insert()'''


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 130-131: truncated \UXXXXXXXX escape (3174479261.py, line 1)

In [18]:
'''import os
import pandas as pd
import pyodbc

# Définir les répertoires source et de sortie
source_folder =  r"C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax"

# Paramètres de connexion
server = 'SARAH_BHSS\\SQLEXPRESS'
database = 'SystemeSuiviProduction'
username = 'sa'
password = 'admin'

# Fonction pour établir la connexion à SQL Server
def connect_to_db():
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password};"
        )
        print("Connexion réussie à la base de données.")
        return conn
    except Exception as e:
        print(f"Erreur lors de la connexion : {e}")
        return None

# Fonction pour insérer un DataFrame dans SQL Server
def insert_data_to_sql(df, table_name, conn):
    try:
        cursor = conn.cursor()
        # Vérifier si le DataFrame est vide
        if df.empty:
            print(f"Le DataFrame est vide. Aucune donnée à insérer pour la table '{table_name}'.")
            return
        # Générer la requête d'insertion SQL
        columns = ', '.join(df.columns)
        placeholders = ', '.join(['?'] * len(df.columns))
        insert_query = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
        # Insérer chaque ligne du DataFrame dans la table
        for _, row in df.iterrows():
            cursor.execute(insert_query, tuple(row))
        conn.commit()
        print(f"Données insérées avec succès dans la table '{table_name}'.")
    except Exception as e:
        print(f"Erreur lors de l'insertion des données : {e}")

# Fonction pour parser les fichiers IDC et LDC
def parse_data_file(file_path):
    try:
        # Lire le fichier avec des tabulations comme séparateurs, en remplaçant les 'NULL' par des chaînes vides
        df = pd.read_csv(file_path, sep='\t', header=None, engine='python', na_values=["NULL"])
        # Vérifier le nombre de colonnes
        print(f"Nombre de colonnes dans le fichier : {df.shape[1]}")
        # Définir les noms des colonnes en fonction du nombre de colonnes
        if df.shape[1] == 7:
            df.columns = ['Date', 'Time', 'Source', 'Action', 'Details', 'Status', 'Type']
        elif df.shape[1] == 6:
            df.columns = ['Date', 'Time', 'Source', 'Action', 'Details', 'Status']
        else:
            print(f"Le fichier '{file_path}' a un nombre de colonnes inattendu.")
            return pd.DataFrame()
        print(f"Fichier '{file_path}' chargé avec succès.")

        # Convertir les colonnes Date et Time en formats appropriés
        df['DateJob'] = pd.to_datetime(df['Date'], format='%d/%m/%Y').dt.date
        df['TimeJob'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time

        # Récupérer d'autres colonnes si nécessaire
        df['MainWPCSS'] = df['Source']
        df['ModeJob'] = df['Action']
        df['IdJob'] = df['Details']
        df['NumJob'] = df['Status']
        df['NumJob'] = '1'
        
        # Rendre 'TypeJob' constant, 'A' si 'ModeJob' est 'A', sinon '0'
        df['TypeJob'] = 'A'
        
        # Sélectionner les colonnes correspondantes à la table JOB433
        df = df[['DateJob', 'TimeJob', 'MainWPCSS', 'ModeJob', 'IdJob', 'NumJob', 'TypeJob']]
        
        # Supprimer les lignes contenant des valeurs manquantes
        df = df.dropna()
        print(f"Fichier '{file_path}' parsé avec succès.")
        return df
    except Exception as e:
        print(f"Erreur lors du parsing du fichier '{file_path}' : {e}")
        return pd.DataFrame()

# Fonction principale pour traiter les fichiers
def process_files_and_insert():
    # Connexion à la base de données
    conn = connect_to_db()
    if not conn:
        return
    # Parcourir les fichiers dans le répertoire source
    for root, dirs, files in os.walk(source_folder):
        for file in files:
            file_path = os.path.join(root, file)
            if file.endswith('.LDC'):  # Vérification insensible à la casse
                print(f"Fichier trouvé : {file_path}")
                try:
                    # Lire le fichier en DataFrame
                    df = parse_data_file(file_path)
                    # Vérifier si le DataFrame est vide
                    if df.empty:
                        print(f"Le fichier '{file_path}' ne contient aucune donnée valide.")
                        continue  # Passer au fichier suivant
                    # Extraire la valeur avant la virgule dans 'IdJob'
                    df['IdJob'] = df['IdJob'].str.split(',').str[0]  # Garde la partie avant la virgule
                    # Ajouter une colonne pour tracer la source des données
                    df['FilePath'] = file_path
                    # Supprimer la colonne 'FilePath' avant l'insertion dans SQL
                    df = df.drop(columns=['FilePath'], errors='ignore')
                    # Insérer les données dans la table SQL
                    table_name = 'JOB'  # Nom de la table cible
                    insert_data_to_sql(df, table_name, conn)
                except Exception as e:
                    print(f"Erreur lors du traitement du fichier '{file_path}' : {e}")
    # Fermer la connexion à la base de données
    conn.close()

# Exécuter le script principal
if __name__ == "__main__":
    process_files_and_insert()'''


Connexion réussie à la base de données.
Fichier trouvé : C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\01\Job.LDC
Nombre de colonnes dans le fichier : 7
Fichier 'C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\01\Job.LDC' chargé avec succès.
Fichier 'C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\01\Job.LDC' parsé avec succès.
Données insérées avec succès dans la table 'JOB'.
Fichier trouvé : C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\04\Job.LDC
Nombre de colonnes dans le fichier : 7
Fichier 'C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\04\Job.LDC' chargé avec succès.
Fichier 'C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\04\Job.LDC' parsé avec succès.
Données insérées avec succ

# Chargement "Producti Original"



In [19]:
'''import os
import pyodbc
import pandas as pd

# Liste des sections à exclure (modifiable selon vos besoins)
excluded_blocks = []  # Exemple : ['Block1', 'Block2']

# Fonction pour échapper les apostrophes dans les chaînes de caractères
def escape_single_quotes(value):
    """Escape single quotes in string values for SQL compatibility."""
    if isinstance(value, str):
        return value.replace("'", "''")
    return value

# Fonction pour parser et combiner les enregistrements
def parse_and_combine_records(file_path, conn_str, table_name):
    """
    Parse the SDC file and combine sections like 'LearnStarted' with the following 'Counter' section,
    then insert the result into an SQL Server table.
    """

    # Extraire le numéro de la machine à partir du chemin
    num_machine = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(file_path))))

    with open(file_path, 'r') as f:
        transactions = []
        current_record = {}
        current_section = None

        for line in f:
            line = line.strip()

            if line.startswith("[") and line.endswith("]"):
                # Ajouter l'enregistrement précédent avant de passer à un nouveau bloc
                if current_record:
                    transactions.append(current_record)

                # Initialiser une nouvelle section
                current_section = line[1:-1]
                current_record = {"Section": current_section}

            elif "=" in line and current_section not in excluded_blocks:
                key, value = line.split("=", 1)
                value = value.strip()

                # Parsing des champs spécifiques
                if key == "DateTimeStamp":
                    date, time = value.split(",")
                    current_record["Date"] = date.strip()
                    current_record["Time"] = time.strip()
                elif "," in value:
                    parts = value.split(",")
                    if key == "Wire":
                        current_record["Wire_Code"] = parts[0].strip()
                        current_record["Wire_Quantity_Global"] = parts[1].strip()
                        current_record["Wire_Good_Quantity"] = parts[2].strip()
                    elif key == "Terminal":
                        terminal_index = len([k for k in current_record.keys() if k.startswith("Terminal_Code")]) + 1
                        if terminal_index == 1:
                            current_record["Terminal_Code_1"] = parts[0].strip()
                            current_record["Terminal_Quantity_Global_1"] = parts[1].strip()
                            current_record["Terminal_Good_Quantity_1"] = parts[2].strip()
                        elif terminal_index == 2:
                            current_record["Terminal_Code_2"] = parts[0].strip()
                            current_record["Terminal_Quantity_Global_2"] = parts[1].strip()
                            current_record["Terminal_Good_Quantity_2"] = parts[2].strip()                    
                    elif key == "Seal":
                        Seal_index = len([k for k in current_record.keys() if k.startswith("Seal_Code")]) + 1
                        if Seal_index == 1:
                            current_record["Seal_Code_1"] = parts[0].strip()
                            current_record["Seal_Quantity_Global_1"] = parts[1].strip()
                            current_record["Seal_Good_Quantity_1"] = parts[2].strip()
                        elif Seal_index == 2:
                            current_record["Seal_Code_2"] = parts[0].strip()
                            current_record["Seal_Quantity_Global_2"] = parts[1].strip()
                            current_record["Seal_Good_Quantity_2"] = parts[2].strip()
                    elif key == "Job":
                        current_record["Job"] = f"{parts[0].strip()},{parts[1].strip()}"
                    elif key == "ProductionPieces":
                        current_record["ProductionGoodPieces"] = parts[0].strip()
                        current_record["ProductionPieces_Demande"] = parts[1].strip()
                else:
                    current_record[key.strip()] = value

        # Ajouter le dernier enregistrement
        if current_record:
            transactions.append(current_record)

    # Fusionner LearnStarted avec le Counter qui suit immédiatement
    merged_transactions = []
    temp_record = {}

    for record in transactions:
        if record["Section"] != "Counter":
            if temp_record:
                merged_transactions.append(temp_record)
            temp_record = record.copy()
        else:
            if temp_record:
                for key, value in record.items():
                    if key not in temp_record or not temp_record[key]:
                        temp_record[key] = value
                merged_transactions.append(temp_record)
                temp_record = {}

    if temp_record:
        merged_transactions.append(temp_record)

    # Ajouter NumMachine à chaque enregistrement
    for record in merged_transactions:
        record["NumMachine"] = num_machine

    # Convertir en DataFrame
    df = pd.DataFrame(merged_transactions)

    # Connexion à la base de données SQL Server
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()

    # Insérer les données dans la table SQL Server
    for index, row in df.iterrows():
        # Créer une requête d'insertion SQL
        columns = ", ".join(df.columns)
        values = ", ".join([f"'{escape_single_quotes(str(val))}'" if val is not None else "NULL" for val in row])
        query = f"INSERT INTO {table_name} ({columns}) VALUES ({values})"

        # Exécuter la requête d'insertion
        cursor.execute(query)

    # Commit les changements
    conn.commit()

    # Fermer la connexion
    cursor.close()
    conn.close()

    print(f"Les données ont été insérées avec succès dans la table '{table_name}'.")
    return df

# Fonction principale pour traiter tous les fichiers
def process_files(source_folder, conn_str, table_name):
    """Process all files in the given source folder and insert their data into SQL."""
    for machine_folder in os.listdir(source_folder):
        machine_path = os.path.join(source_folder, machine_folder)
        if os.path.isdir(machine_path):
            for month_folder in os.listdir(machine_path):
                month_path = os.path.join(machine_path, month_folder)
                if os.path.isdir(month_path):
                    for day_folder in os.listdir(month_path):
                        day_path = os.path.join(month_path, day_folder)
                        if os.path.isdir(day_path):
                            for file in os.listdir(day_path):
                                file_path = os.path.join(day_path, file)
                                if file.lower().endswith('.sdc'):
                                    print(f"Processing file: {file_path}")
                                    df = parse_and_combine_records(file_path, conn_str, table_name)
                                    print(f"File {file_path} data inserted successfully.")

source_folder = r"C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax"
server = 'SARAH_BHSS\\SQLEXPRESS'
database = 'SystemeSuiviProduction'
username = 'sa'
password = 'admin'

conn_str = (
    f"Driver={{ODBC Driver 17 for SQL Server}};"
    f"Server={server};"
    f"Database={database};"
    f"UID={username};"
    f"PWD={password};"
)
table_name = "ProductiRecords"

process_files(source_folder, conn_str, table_name)'''

Processing file: C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\01\Producti.SDC
Les données ont été insérées avec succès dans la table 'ProductiRecords'.
File C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\01\Producti.SDC data inserted successfully.
Processing file: C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\04\Producti.SDC
Les données ont été insérées avec succès dans la table 'ProductiRecords'.
File C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\04\Producti.SDC data inserted successfully.
Processing file: C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_komax\433\11\05\Producti.SDC
Les données ont été insérées avec succès dans la table 'ProductiRecords'.
File C:\Users\sarah\Downloads\Semestre1_DSB1\sem1\ADVANCED DATAWAREHOUSE\Projet 2024\Data_k

----
#                                                                            Transformation
---

# Transform PRODUCTI => Suppresion Lignes inutiles et faire calcul necessaire 

In [11]:
import pyodbc
import pandas as pd

# Paramètres de connexion
server = "localhost"
database = "TransformSystemeSuiviProduction"
username = "sa"  # Super administrateur SQL
password = "admin"

# Fonction pour établir la connexion à SQL Server
def connect_to_db():
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password};"
            f"Trusted_Connection=no;"  # Assure-toi de ne pas utiliser Windows Authentication
        )
        print("Connexion réussie à la base de données.")
        return conn
    except Exception as e:
        print(f"Erreur lors de la connexion : {e}")
        return None

# Requête SQL pour récupérer **toutes les lignes**
query = """
SELECT * 
FROM ProductiTransform
"""

# Blocs à exclure
excluded_blocks = ["MaterialChangeDetection", "MeasurementsLeadSet1", "QualityParameters", "MeasurementData"]

# Connexion à la base de données
conn = connect_to_db()

if conn:
    try:
        # Exécuter la requête et récupérer toutes les lignes dans un DataFrame
        df = pd.read_sql(query, conn)
        print("Toutes les données récupérées de la base de données.")

        # Filtrer les lignes pour exclure les sections spécifiées
        df_filtered = df[~df["Section"].isin(excluded_blocks)]
        print("Données après exclusion des blocs :")
        print(df_filtered)

        # Convertir les colonnes pertinentes en numérique
        columns_to_convert = [
            "Wire_Quantity_Global", "Wire_Good_Quantity",
            "Terminal_Quantity_Global_1", "Terminal_Good_Quantity_1",
            "Terminal_Quantity_Global_2", "Terminal_Good_Quantity_2",
            "Seal_Quantity_Global_1", "Seal_Good_Quantity_1",
            "Seal_Quantity_Global_2", "Seal_Good_Quantity_2"
        ]

        # Assurer que les colonnes pertinentes sont converties en numériques
        for col in columns_to_convert:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors="coerce")

        # Résultats combinés après traitement
        combined_data = []

        for index in range(len(df_filtered) - 1):
            current_row = df_filtered.iloc[index]
            next_row = df_filtered.iloc[index + 1]

            current_section = current_row["Section"]
            next_section = next_row["Section"]
            next_section_Time = next_row["Time"]

            # On garde la logique pour 'Started' et 'Restarted'
            if "Started" in current_section or "Restarted" in current_section:
                base_name = current_section.replace("Started", "").replace("Restarted", "")
                new_wire_quantity = next_row["Wire_Quantity_Global"] - current_row["Wire_Quantity_Global"]
                new_wire_good_quantity = next_row["Wire_Good_Quantity"] - current_row["Wire_Good_Quantity"]

                # Calcul des nouvelles quantités pour chaque terminal/Seal
                new_terminal_quantity_1 = next_row["Terminal_Quantity_Global_1"] - current_row["Terminal_Quantity_Global_1"]
                new_terminal_good_quantity_1 = next_row["Terminal_Good_Quantity_1"] - current_row["Terminal_Good_Quantity_1"]
                new_terminal_quantity_2 = next_row["Terminal_Quantity_Global_2"] - current_row["Terminal_Quantity_Global_2"]
                new_terminal_good_quantity_2 = next_row["Terminal_Good_Quantity_2"] - current_row["Terminal_Good_Quantity_2"]

                new_seal_quantity_1 = next_row["Seal_Quantity_Global_1"] - current_row["Seal_Quantity_Global_1"]
                new_seal_good_quantity_1 = next_row["Seal_Good_Quantity_1"] - current_row["Seal_Good_Quantity_1"]
                new_seal_quantity_2 = next_row["Seal_Quantity_Global_2"] - current_row["Seal_Quantity_Global_2"]
                new_seal_good_quantity_2 = next_row["Seal_Good_Quantity_2"] - current_row["Seal_Good_Quantity_2"]

                # Mettre à jour la ligne avec les nouvelles valeurs
                current_row["Wire_Quantity_Global"] = new_wire_quantity
                current_row["Wire_Good_Quantity"] = new_wire_good_quantity

                current_row["Terminal_Quantity_Global_1"] = new_terminal_quantity_1
                current_row["Terminal_Good_Quantity_1"] = new_terminal_good_quantity_1

                current_row["Terminal_Quantity_Global_2"] = new_terminal_quantity_2
                current_row["Terminal_Good_Quantity_2"] = new_terminal_good_quantity_2

                current_row["Seal_Quantity_Global_1"] = new_seal_quantity_1
                current_row["Seal_Good_Quantity_1"] = new_seal_good_quantity_1

                current_row["Seal_Quantity_Global_2"] = new_seal_quantity_2
                current_row["Seal_Good_Quantity_2"] = new_seal_good_quantity_2

                current_row["Section"] = next_section  # Mettre à jour la section avec la suivante
                current_row["Time"] = next_section_Time  # Mettre à jour l'heure avec celle de la section suivante

                combined_data.append(current_row)

        # Créer un DataFrame avec les résultats combinés
        result_df = pd.DataFrame(combined_data)
        print("Résultat combiné après traitement :")
        print(result_df)

    except Exception as e:
        print(f"Erreur lors de l'exécution de la requête : {e}")
    finally:
        conn.close()  # Toujours fermer la connexion
        print("Connexion fermée.")
else:
    print("Impossible de se connecter à la base de données.")


Connexion réussie à la base de données.


C:\Users\sarah\AppData\Local\Temp\ipykernel_35404\71390211.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Toutes les données récupérées de la base de données.
Données après exclusion des blocs :
                Section        Date      Time ArticleKey  UserName  \
0         SampleStarted  01/11/2024  06:31:28   57628_-2  MED AMIN   
1      SampleTerminated  01/11/2024  06:31:40   57628_-2  MED AMIN   
2          LearnStarted  01/11/2024  06:32:19   57628_-2  MED AMIN   
3          LearnAborted  01/11/2024  06:32:30   57628_-2  MED AMIN   
4         SampleStarted  01/11/2024  06:36:26   57628_-2  MED AMIN   
...                 ...         ...       ...        ...       ...   
17119  SampleTerminated  15/11/2024  22:32:36   57634_-7   HAITHEM   
17120      LearnStarted  15/11/2024  22:32:44   57634_-7   HAITHEM   
17123   LearnTerminated  15/11/2024  22:33:28   57634_-7   HAITHEM   
17124     SampleStarted  15/11/2024  22:33:34   57634_-7   HAITHEM   
17125  SampleTerminated  15/11/2024  22:38:18   57634_-7   HAITHEM   

          Wire_Code Wire_Quantity_Global Wire_Good_Quantity Terminal_C

C:\Users\sarah\AppData\Local\Temp\ipykernel_35404\71390211.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered[col] = pd.to_numeric(df_filtered[col], errors="coerce")
C:\Users\sarah\AppData\Local\Temp\ipykernel_35404\71390211.py:92: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_row["Wire_Quantity_Global"] = new_wire_quantity
C:\Users\sarah\AppData\Local\Temp\ipykernel_35404\71390211.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org

Résultat combiné après traitement :
                Section        Date      Time ArticleKey  UserName  \
0      SampleTerminated  01/11/2024  06:31:40   57628_-2  MED AMIN   
2          LearnAborted  01/11/2024  06:32:30   57628_-2  MED AMIN   
4      SampleTerminated  01/11/2024  06:36:38   57628_-2  MED AMIN   
6      SampleTerminated  01/11/2024  06:38:11   57628_-2  MED AMIN   
8       LearnTerminated  01/11/2024  06:39:47   57628_-2  MED AMIN   
...                 ...         ...       ...        ...       ...   
17114  SampleTerminated  15/11/2024  22:28:35   57634_-7   HAITHEM   
17116  SampleTerminated  15/11/2024  22:30:55   57634_-7   HAITHEM   
17118  SampleTerminated  15/11/2024  22:32:36   57634_-7   HAITHEM   
17120   LearnTerminated  15/11/2024  22:33:28   57634_-7   HAITHEM   
17124  SampleTerminated  15/11/2024  22:38:18   57634_-7   HAITHEM   

          Wire_Code  Wire_Quantity_Global  Wire_Good_Quantity Terminal_Code_1  \
0         CU04CR001                 320.0 

In [12]:
result_df

,Section,Date,Time,ArticleKey,UserName,Wire_Code,Wire_Quantity_Global,Wire_Good_Quantity,Terminal_Code_1,Terminal_Quantity_Global_1,...,ProductionRequestedPieces,ProductionPieces,UserRequestedPieces,TotalGoodPieces,Seal_Code_1,Seal_Code_2,Seal_Quantity_Global_1,Seal_Good_Quantity_1,Seal_Quantity_Global_2,Seal_Good_Quantity_2
0,SampleTerminated,01/11/2024,06:31:40,57628_-2,MED AMIN,CU04CR001,320.0,0.0,T5556-T,1.0,...,nan,None,nan,nan,None,None,NaN,NaN,NaN,NaN
2,LearnAborted,01/11/2024,06:32:30,57628_-2,MED AMIN,CU04CR001,0.0,0.0,T5556-T,0.0,...,nan,None,nan,nan,None,None,NaN,NaN,NaN,NaN
4,SampleTerminated,01/11/2024,06:36:38,57628_-2,MED AMIN,CU04CR001,320.0,0.0,T5556-T,1.0,...,nan,None,nan,nan,None,None,NaN,NaN,NaN,NaN
6,SampleTerminated,01/11/2024,06:38:11,57628_-2,MED AMIN,CU04CR001,320.0,0.0,T5556-T,1.0,...,nan,None,nan,nan,None,None,NaN,NaN,NaN,NaN
8,LearnTerminated,01/11/2024,06:39:47,57628_-2,MED AMIN,CU04CR001,620.0,0.0,T5556-T,3.0,...,nan,None,nan,nan,None,None,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17114,SampleTerminated,15/11/2024,22:28:35,57634_-7,HAITHEM,CU02CR002/F3,125.0,120.0,T7-1452665-1,1.0,...,nan,nan,nan,nan,Y2-GO967067-2,None,1.0,1.0,NaN,NaN
17116,SampleTerminated,15/11/2024,22:30:55,57634_-7,HAITHEM,CU02CR002/F3,125.0,120.0,T7-1452665-1,1.0,...,nan,nan,nan,nan,Y2-GO967067-2,None,1.0,1.0,NaN,NaN
17118,SampleTerminated,15/11/2024,22:32:36,57634_-7,HAITHEM,CU02CR002/F3,125.0,120.0,T7-1452665-1,1.0,...,nan,nan,nan,nan,Y2-GO967067-2,None,1.0,1.0,NaN,NaN
17120,LearnTerminated,15/11/2024,22:33:28,57634_-7,HAITHEM,CU02CR002/F3,245.0,120.0,T7-1452665-1,1.0,...,nan,nan,nan,nan,Y2-GO967067-2,None,2.0,1.0,NaN,NaN


In [13]:
nombre_de_lignes = len(result_df)
print(f"Nombre de lignes avec len : {nombre_de_lignes}")

Nombre de lignes avec len : 5980


In [29]:
'''import pyodbc
import pandas as pd

# Paramètres de connexion
server = "localhost"
database = "TransformSystemeSuiviProduction"
username = "sa"  # Super administrateur SQL
password = "admin"

# Fonction pour établir la connexion à SQL Server
def connect_to_db():
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password};"
            f"Trusted_Connection=no;"
        )
        print("Connexion réussie à la base de données.")
        return conn
    except Exception as e:
        print(f"Erreur lors de la connexion : {e}")
        return None

# Fonction de nettoyage des données
def clean_data(df):
    # Colonnes numériques : convertir en float, remplacer NaN par 0
    numeric_cols = [
        "Wire_Quantity_Global", "Wire_Good_Quantity", 
        "Terminal_Quantity_Global_1", "Terminal_Good_Quantity_1",
        "Terminal_Quantity_Global_2", "Terminal_Good_Quantity_2",
        "SampleRequestedPieces", "ProductionRequestedPieces",
        "ProductionPieces", "UserRequestedPieces", "TotalGoodPieces",
        "Seal_Quantity_Global_1", "Seal_Good_Quantity_1",
        "Seal_Quantity_Global_2", "Seal_Good_Quantity_2"
    ]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)  # Convertir en float, NaN -> 0

    # Colonnes de texte : remplacer NaN par des chaînes vides
    text_cols = [
        "Section", "ArticleKey", "UserName", "Wire_Code", 
        "Terminal_Code_1", "Terminal_Code_2", 
        "Seal_Code_1", "Seal_Code_2", "NumMachine"
    ]
    for col in text_cols:
        df[col] = df[col].fillna("")

    # Colonnes date et heure : s'assurer du bon format
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").fillna(pd.Timestamp("1970-01-01"))
    df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce").dt.time.fillna("00:00:00")
    
    return df

# Connexion et insertion
conn = connect_to_db()
if conn:
    cursor = conn.cursor()

    try:
        # Nettoyage des données
        result_df = clean_data(result_df)  # Applique le nettoyage au DataFrame

        # Boucle pour insérer chaque ligne
        for _, row in result_df.iterrows():
            cursor.execute("""
                INSERT INTO ProductiTransfomCalculated (
                    Section, Date, Time, ArticleKey, UserName,
                    Wire_Code, Wire_Quantity_Global, Wire_Good_Quantity,
                    Terminal_Code_1, Terminal_Quantity_Global_1, Terminal_Good_Quantity_1,
                    Terminal_Code_2, Terminal_Quantity_Global_2, Terminal_Good_Quantity_2,
                    NumMachine, SampleRequestedPieces, ProductionRequestedPieces,
                    ProductionPieces, UserRequestedPieces, TotalGoodPieces,
                    Seal_Code_1, Seal_Code_2, Seal_Quantity_Global_1, Seal_Good_Quantity_1,
                    Seal_Quantity_Global_2, Seal_Good_Quantity_2
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row["Section"], row["Date"], row["Time"], row["ArticleKey"], row["UserName"],
                row["Wire_Code"], row["Wire_Quantity_Global"], row["Wire_Good_Quantity"],
                row["Terminal_Code_1"], row["Terminal_Quantity_Global_1"], row["Terminal_Good_Quantity_1"],
                row["Terminal_Code_2"], row["Terminal_Quantity_Global_2"], row["Terminal_Good_Quantity_2"],
                row["NumMachine"], row["SampleRequestedPieces"], row["ProductionRequestedPieces"],
                row["ProductionPieces"], row["UserRequestedPieces"], row["TotalGoodPieces"],
                row["Seal_Code_1"], row["Seal_Code_2"], row["Seal_Quantity_Global_1"], row["Seal_Good_Quantity_1"],
                row["Seal_Quantity_Global_2"], row["Seal_Good_Quantity_2"]
            ))

        conn.commit()
        print("Données insérées avec succès dans ProductiTransfomCalculated.")

    except Exception as e:
        print(f"Erreur lors de l'insertion des données : {e}")
    finally:
        cursor.close()
        conn.close()
        print("Connexion fermée.")
else:
    print("Impossible de se connecter à la base de données.")'''


'import pyodbc\nimport pandas as pd\n\n# Paramètres de connexion\nserver = "localhost"\ndatabase = "TransformSystemeSuiviProduction"\nusername = "sa"  # Super administrateur SQL\npassword = "admin"\n\n# Fonction pour établir la connexion à SQL Server\ndef connect_to_db():\n    try:\n        conn = pyodbc.connect(\n            f"DRIVER={{ODBC Driver 17 for SQL Server}};"\n            f"SERVER={server};"\n            f"DATABASE={database};"\n            f"UID={username};"\n            f"PWD={password};"\n            f"Trusted_Connection=no;"\n        )\n        print("Connexion réussie à la base de données.")\n        return conn\n    except Exception as e:\n        print(f"Erreur lors de la connexion : {e}")\n        return None\n\n# Fonction de nettoyage des données\ndef clean_data(df):\n    # Colonnes numériques : convertir en float, remplacer NaN par 0\n    numeric_cols = [\n        "Wire_Quantity_Global", "Wire_Good_Quantity", \n        "Terminal_Quantity_Global_1", "Terminal_Good_Q

# Creation DIMENSION : DimJOB

In [6]:
'''import pyodbc
import pandas as pd

# Paramètres de connexion
source_server = "localhost"
source_database = "TransformSystemeSuiviProduction"
destination_server = "localhost"
destination_database = "DWHSystemeSuiviProduction"
username = "sa"
password = "admin"

# Fonction pour établir la connexion à une base SQL Server
def connect_to_db(server, database):
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password};"
            f"Trusted_Connection=no;"
        )
        print(f"Connexion réussie à la base de données : {database}")
        return conn
    except Exception as e:
        print(f"Erreur lors de la connexion à {database} : {e}")
        return None

# Requête SQL pour extraire les colonnes spécifiques
query = """
SELECT DISTINCT 
    [DateJob],
    [TimeJob],
    [IdJob],
    [ModeJob]
FROM TableGlobalPRODUCTION;
"""

# Connexion aux bases de données source et destination
source_conn = connect_to_db(source_server, source_database)
destination_conn = connect_to_db(destination_server, destination_database)

if source_conn and destination_conn:
    try:
        # Extraction des données depuis la base source
        df = pd.read_sql(query, source_conn)
        print("Données extraites de la base source :")
        print(df.head())

        # Supprimer les doublons
        df = df.drop_duplicates()

        # Création de la table DimJob dans la base destination avec auto-incrémentation
        create_table_query = """
        IF OBJECT_ID('DimJob', 'U') IS NOT NULL
        BEGIN
            DROP TABLE DimJob;
        END;

        CREATE TABLE DimJob (
            PkJob INT IDENTITY(1,1) PRIMARY KEY,
            DateJob DATE,
            TimeJob TIME,
            IdJob NVARCHAR(50),
            ModeJob NVARCHAR(50)
        );
        """
        destination_cursor = destination_conn.cursor()
        destination_cursor.execute(create_table_query)
        destination_conn.commit()
        print("Table DimJob recréée dans la base destination.")

        # Insertion des données dans la table DimJob
        for _, row in df.iterrows():
            insert_query = """
            INSERT INTO DimJob (DateJob, TimeJob, IdJob, ModeJob)
            VALUES (?, ?, ?, ?);
            """
            destination_cursor.execute(insert_query, row['DateJob'], row['TimeJob'], row['IdJob'], row['ModeJob'])

        destination_conn.commit()
        print("Données insérées dans DimJob avec succès.")

    except Exception as e:
        print(f"Erreur lors de l'extraction ou de l'insertion : {e}")

    finally:
        # Fermer les connexions
        source_conn.close()
        destination_conn.close()
        print("Connexions fermées.")'''


Connexion réussie à la base de données : TransformSystemeSuiviProduction
Connexion réussie à la base de données : DWHSystemeSuiviProduction
Données extraites de la base source :
      DateJob           TimeJob      IdJob ModeJob
0  2024-11-11  08:51:30.0000000  57479_-11  delete
1  2024-11-11  08:50:50.0000000  57479_-16  delete
2  2024-11-11  08:50:40.0000000  57479_-17  delete
3  2024-11-11  13:27:45.0000000  57479_-13  insert
4  2024-11-11  08:50:15.0000000  57479_-20  delete
Table DimJob recréée dans la base destination.
Données insérées dans DimJob avec succès.
Connexions fermées.


C:\Users\sarah\AppData\Local\Temp\ipykernel_28336\3830572402.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, source_conn)
